# image-agent — ローカル画像生成

**重要な制約（LoRAなし）**
- このエージェントはLoRAを使用しません。顔・髪・衣装の完全な一貫性は保証されません。
- 参考画像は分析に使用しますが、ControlNet等によるピクセルレベルの転写は行いません。
- AI が推測した内容（`inferred`）はユーザー指定内容とは区別して表示されます。

**実機未検証事項**
- RTX 2080での全体パイプライン（mockテストのみ確認済み）
- FLUX.3 INT8 ConvRot（EXPERIMENTAL — 未検証）
- 768×432 / 432×768解像度でのVRAM収束

**使い方**
1. `dataset/` にMarkdownと参考画像を置く（またはこのNotebookからアップロード）
2. セル2で環境確認
3. セル3でMarkdownパスを設定
4. セル4で計画のみ実行してプランを確認
5. セル5で本生成を実行

In [ ]:
# セル1: 環境確認
import sys
print('Python:', sys.version)

try:
    import torch
    print('PyTorch:', torch.__version__)
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        vram_mib = props.total_memory // (1024 * 1024)
        print(f'GPU: {props.name}, VRAM: {vram_mib} MiB')
        if vram_mib < 7500:
            print('WARNING: VRAM < 7500 MiB — generation may OOM')
    else:
        print('CUDA: not available — will use mock backend')
except ImportError:
    print('torch not installed — only mock backend available')

try:
    from image_agent.runtime.resources import check_resources
    report = check_resources()
    if report.ram_available_gib:
        print(f'RAM available: {report.ram_available_gib:.1f} GiB')
    if report.disk_free_gib:
        print(f'Disk free: {report.disk_free_gib:.1f} GiB')
    for w in report.warnings:
        print(f'  WARNING: {w}')
except Exception as e:
    print(f'Resource check error: {e}')

import os
dataset_root = '/workspace/dataset'
if os.path.exists(dataset_root):
    files = list(os.listdir(dataset_root))
    print(f'\ndataset/ ({len(files)} items):', files[:10])
else:
    print('\nWARNING: /workspace/dataset does not exist')

In [ ]:
# セル2: datasetへのファイルアップロード（オプション）
# Jupyterのファイルブラウザからもアップロード可能です。
import os
os.makedirs('/workspace/dataset', exist_ok=True)

# ファイルアップロードウィジェット（ipywidgets が必要）
try:
    import ipywidgets as widgets
    from IPython.display import display
    
    uploader = widgets.FileUpload(accept='', multiple=True)
    btn = widgets.Button(description='Upload to /workspace/dataset/')
    output = widgets.Output()

    def on_upload(b):
        with output:
            from pathlib import Path
            for info in uploader.value:
                name = Path(info['name']).name
                dest = f'/workspace/dataset/{name}'
                with open(dest, 'wb') as f:
                    f.write(info['content'])
                print(f'Saved: {dest}')

    btn.on_click(on_upload)
    display(uploader, btn, output)
except ImportError:
    print('ipywidgets not available. Use Jupyter file browser or docker cp to upload files.')
    print('  docker cp ./dataset/ <container>:/workspace/dataset/')

In [ ]:
# セル3: 設定
# ここで生成パラメータを調整してください。

MARKDOWN_PATH = '/workspace/dataset/request.md'  # <- 変更してください
BACKEND = 'flux2'       # 'flux2', 'flux3' (experimental), 'mock' (GPUなし)
SEED = None             # None でランダム、整数で固定
WIDTH = 512             # RTX 2080 安全な初期値
HEIGHT = 512
result = None

# FLUX.3 ライセンス注意（バックエンドを変更する場合）
if BACKEND == 'flux3':
    print('NOTE: FLUX.3 uses FLUX Kommunity License (not Apache-2.0).')
    print('      Review the license at https://bfl.ai before proceeding.')
    print('      FLUX.3 on RTX 2080 is EXPERIMENTAL and unverified.')

# GPUがなければmockにフォールバック
try:
    import torch
    if not torch.cuda.is_available() and BACKEND != 'mock':
        print(f'No GPU detected. Switching backend to mock (was: {BACKEND})')
        BACKEND = 'mock'
except ImportError:
    BACKEND = 'mock'

print(f'Config: backend={BACKEND}, size={WIDTH}x{HEIGHT}, seed={SEED}')
print(f'Markdown: {MARKDOWN_PATH}')

In [ ]:
# セル4: 計画のみ実行（生成なし）
from image_agent.notebook import ImageAgentSession

session = ImageAgentSession(
    prompt=MARKDOWN_PATH,
    backend=BACKEND,
    seed=SEED,
    width=WIDTH,
    height=HEIGHT,
)

plan = session.plan()
if plan is not None:
    session.show_plan()
    session.show_constraints()
    print('\nプランを確認してください。生成を続ける場合は次のセルを実行してください。')
else:
    print('プランニングに失敗しました。エラーを確認してください。')

In [ ]:
# セル5: 本生成
# このセルは時間がかかります（モデルのダウンロードを含む）。
# Qwen → FLUX.2 → Critic → Real-ESRGAN の順で実行されます。
# 各モデルは別プロセスで実行され、終了時にVRAMを解放します。

if plan is None:
    print('計画がないため生成できません。セル4を先に実行してください。')
else:
    result = session.run()
    session.show_result()

In [ ]:
# セル6: 生成結果とCritic評価の詳細表示
if result is not None and result.critic_results:
    for i, cr in enumerate(result.critic_results):
        print(f'\nCritic評価 {i+1}:')
        print(f'  Action: {cr.action.value}')
        print(f'  Overall score: {cr.overall_score:.2f}')
        print(f'  Confidence: {cr.confidence:.2f}')
        print(f'  Rationale: {cr.rationale}')
        if cr.must_preserve_violations:
            print(f'  Violations: {cr.must_preserve_violations}')
        if cr.suggested_revision:
            print(f'  Suggested revision: {cr.suggested_revision}')
        print()
        if cr.confidence < 0.5:
            print('  NOTE: Low confidence — human review recommended.')
else:
    print('No critic results available.')

In [ ]:
# セル7: アップスケール情報
if result is not None and result.upscale_result:
    ur = result.upscale_result
    print('アップスケール結果:')
    print(f'  入力: {ur.input_width}x{ur.input_height}')
    print(f'  出力: {ur.output_width}x{ur.output_height}')
    print(f'  モデル: {ur.model_name}')
    print(f'  Tile: {ur.tile_size}px')
    if ur.oom_fallback_used:
        print('  NOTE: OOMによりtile sizeを縮小しました（128px）')
    if ur.output_asset_ids:
        for path in ur.output_asset_ids:
            if path.endswith('.png'):
                try:
                    from IPython.display import Image, display
                    print(f'  {path}')
                    display(Image(path, width=600))
                except Exception:
                    print(f'  出力: {path}')
else:
    print('アップスケール結果なし（生成が完了していないか、アップスケールが失敗した可能性があります）')

In [ ]:
# セル8: Run一式のダウンロード
if result is not None:
    session.download_run()
else:
    print('実行結果がありません。')

In [ ]:
# セル9: 再開（前回のrunを引き継ぐ場合）
# RESUME_RUN_ID に run_YYYYMMDDTHHMMSS_xxxxxxxx 形式のIDを指定してください。

RESUME_RUN_ID = None  # 例: 'run_20260101T120000_abcdef12'

if RESUME_RUN_ID:
    resume_session = ImageAgentSession(
        prompt=MARKDOWN_PATH,
        backend=BACKEND,
        seed=SEED,
        width=WIDTH,
        height=HEIGHT,
        resume_run_id=RESUME_RUN_ID,
    )
    result = resume_session.run()
    resume_session.show_result()
else:
    print('RESUME_RUN_ID を設定して実行してください。')